# Step 1 — Build the City List (population ≥ 100,000 in 2020)

## Purpose
Define the universe of US cities in scope for the analysis. Every subsequent step operates
on this list as its sample frame.

## Why population ≥ 100,000?
O'Byrne (2015) — the only systematic academic source on 311 adoption — covers only cities
with population > 100,000 in 2003. Below that threshold we have no systematic
adopter/non-adopter classification. Including smaller cities would add rows with unknown
status that cannot be resolved without a full city-by-city search, introducing
misclassification bias in Part 3.

## Why 2020 as the population timestamp?
1. The Census PEP sub-county file (`sub-est2023.csv`) starts from 2020 — it is the earliest
   year available in this file series.
2. 2020 is the most recent decennial Census base year, giving the most accurate population
   estimates for current city boundaries.
3. Using 2020 captures fast-growing cities that crossed 100k after O'Byrne's 2003 cutoff.

**Caveat:** Cities that were above 100k in 2003 but shrank below by 2020 will be excluded.
This is a known limitation flagged in the analysis.

## Data sources
- `data_raw/sub-est2023.csv` — Census PEP sub-county totals (population filter)
- `data_raw/2023_Gaz_place_national.txt` — Census Gazetteer 2023 (geographic attributes)

## Output
`data_clean/city_list.csv`

In [1]:
import pandas as pd

RAW   = '../data_raw/'
CLEAN = '../data_clean/'
POP_THRESHOLD = 100_000

## 1. Load and filter Census PEP sub-county population estimates

In [2]:
pep = pd.read_csv(
    RAW + 'sub-est2023.csv',
    encoding='latin-1',
    dtype={'STATE': str, 'COUNTY': str, 'PLACE': str},
)

print(f'PEP raw rows: {len(pep):,}')
print(f'SUMLEV values: {sorted(pep["SUMLEV"].unique())}')
print(f'FUNCSTAT values: {sorted(pep["FUNCSTAT"].dropna().unique())}')

PEP raw rows: 81,374
SUMLEV values: [np.int64(40), np.int64(50), np.int64(61), np.int64(71), np.int64(157), np.int64(162), np.int64(170), np.int64(172)]
FUNCSTAT values: ['A', 'B', 'C', 'F', 'G', 'I', 'N', 'S']


In [3]:
# Filter: incorporated places (SUMLEV=162) that are active governments (FUNCSTAT=A)
places = pep[
    (pep['SUMLEV'] == 162) &
    (pep['FUNCSTAT'] == 'A') &
    (pep['POPESTIMATE2020'] >= POP_THRESHOLD)
].copy()

# Build 7-digit GEOID: 2-digit state FIPS + 5-digit place FIPS
places['GEOID'] = places['STATE'].str.zfill(2) + places['PLACE'].str.zfill(5)

# Rename and keep needed columns
places = places.rename(columns={
    'NAME':            'place_name',
    'STNAME':          'state_name',
    'POPESTIMATE2020': 'pop_2020',
    'POPESTIMATE2021': 'pop_2021',
    'POPESTIMATE2022': 'pop_2022',
    'POPESTIMATE2023': 'pop_2023',
})[['GEOID', 'place_name', 'state_name', 'pop_2020', 'pop_2021', 'pop_2022', 'pop_2023']]

print(f'Incorporated places with pop >= {POP_THRESHOLD:,}: {len(places)}')
print(f'Duplicate GEOIDs: {places["GEOID"].duplicated().sum()}')

Incorporated places with pop >= 100,000: 314
Duplicate GEOIDs: 0


## 2. Load Census Gazetteer — geographic attributes

In [4]:
gaz = pd.read_csv(
    RAW + '2023_Gaz_place_national.txt',
    sep='\t',
    dtype={'GEOID': str},
    encoding='latin-1',
)
gaz.columns = gaz.columns.str.strip()
gaz['GEOID'] = gaz['GEOID'].str.strip().str.zfill(7)

gaz = gaz[['GEOID', 'USPS', 'LSAD', 'FUNCSTAT', 'ALAND_SQMI', 'INTPTLAT', 'INTPTLONG']].rename(columns={
    'USPS':       'state_abbr',
    'LSAD':       'place_type',
    'FUNCSTAT':   'funcstat',
    'ALAND_SQMI': 'land_area_sqmi',
    'INTPTLAT':   'lat',
    'INTPTLONG':  'lon',
})

print(f'Gazetteer rows: {len(gaz):,}')

Gazetteer rows: 32,329


## 3. Join PEP + Gazetteer on GEOID

In [5]:
city_list = places.merge(gaz, on='GEOID', how='left')

# Reorder columns to match plan spec
col_order = [
    'GEOID', 'place_name', 'state_abbr', 'state_name',
    'pop_2020', 'pop_2021', 'pop_2022', 'pop_2023',
    'lat', 'lon', 'land_area_sqmi', 'place_type', 'funcstat',
]
city_list = city_list[col_order].sort_values('pop_2020', ascending=False).reset_index(drop=True)

# Sanity checks
missing_geo = city_list['lat'].isna().sum()
print(f'Cities in working sample : {len(city_list)}')
print(f'Missing Gazetteer match  : {missing_geo}')
print(f'States represented       : {city_list["state_abbr"].nunique()}')
print()
print('Largest 10 cities:')
print(city_list[['place_name', 'state_abbr', 'pop_2020']].head(10).to_string(index=False))
print()
print('Smallest 5 cities in sample:')
print(city_list[['place_name', 'state_abbr', 'pop_2020']].tail(5).to_string(index=False))

Cities in working sample : 314
Missing Gazetteer match  : 0
States represented       : 44

Largest 10 cities:
       place_name state_abbr  pop_2020
    New York city         NY   8740292
 Los Angeles city         CA   3895848
     Chicago city         IL   2743329
     Houston city         TX   2299269
     Phoenix city         AZ   1612459
Philadelphia city         PA   1600684
 San Antonio city         TX   1439257
   San Diego city         CA   1386292
      Dallas city         TX   1303212
    San Jose city         CA   1009319

Smallest 5 cities in sample:
      place_name state_abbr  pop_2020
       Lynn city         MA    101103
New Bedford city         MA    100993
Federal Way city         WA    100988
   Edinburg city         TX    100711
 San Angelo city         TX    100066


## 4. Save

In [6]:
out_path = CLEAN + 'city_list.csv'
city_list.to_csv(out_path, index=False)
print(f'Saved {len(city_list)} rows → {out_path}')
print()
print('Column summary:')
print(city_list.dtypes.to_string())

Saved 314 rows → ../data_clean/city_list.csv

Column summary:
GEOID              object
place_name         object
state_abbr         object
state_name         object
pop_2020            int64
pop_2021            int64
pop_2022            int64
pop_2023            int64
lat               float64
lon               float64
land_area_sqmi    float64
place_type         object
funcstat           object
